# Experiment 5: Subword Tokenization and POS Tagging

This notebook covers:
1. BPE tokenization using a pretrained tokenizer
2. BPE implemented from scratch
3. SentencePiece-based tokenization using a pretrained model
4. Training a SentencePiece BPE tokenizer
5. POS tagging using spaCy
6. POS tagging using NLTK
7. Unique tokens, POS descriptions, and token frequencies using spaCy

**Input file:** `input_sub_word_data.txt`

Keep the input text file in the same directory as this notebook. If it is not found, the notebook uses a short fallback corpus so the examples can still run.

## Setup: Install the required libraries

Run this cell if the libraries are not installed. The spaCy English model and NLTK resources are downloaded in later cells when needed.

In [1]:
%pip install -q transformers sentencepiece spacy nltk

Note: you may need to restart the kernel to use updated packages.


## Load the input corpus

In [1]:
from pathlib import Path

input_path = Path("input_sub_word_data.txt")

if input_path.exists():
    text = input_path.read_text(encoding="utf-8")
    print(f"Loaded corpus from: {input_path.resolve()}")
else:
    text = """
    Natural language processing is a branch of artificial intelligence that
    enables computers to understand, interpret, and generate human language.
    Byte Pair Encoding is a popular subword tokenization algorithm.
    Transformers use attention mechanisms to identify relationships between tokens.
    Machine learning algorithms require numerical representations of text.
    """
    print("input_sub_word_data.txt was not found; using the fallback corpus.")

print("Character count:", len(text))
print("\nPreview:\n")
print(text[:1000])

Loaded corpus from: /Users/saiuppalapati/CL - NLP/Experiment  - 5/input_sub_word_data.txt
Character count: 6682

Preview:

Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization is one of the fundamental steps in natural language processing.
A tokenizer divides text into smaller units called tokens. Tokens may be
words, characters, or subword units. Word-level tokenization is simple and
easy to understand, but it may have difficulty handling rare and unknown
words.

Subword-level tokenization provides an effective solution to the problem
of rare words. Instead of representing every complete word as a separate
token, a subword tokenizer represents a word using smaller meaningful units.
For example, the word playing can be represented as play and i

## 1.1 - BPE tokenization using a pretrained tokenizer

This section loads the pretrained GPT-2 tokenizer. It tokenizes the corpus into subword tokens and converts those tokens to IDs. The first run may download the tokenizer files.

In [2]:
from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

bpe_tokens = gpt2_tokenizer.tokenize(text)
bpe_token_ids = gpt2_tokenizer.convert_tokens_to_ids(bpe_tokens)

print("First 100 BPE tokens:")
print(bpe_tokens[:100])

print("\nFirst 100 token IDs:")
print(bpe_token_ids[:100])

print("\nNumber of tokens:", len(bpe_tokens))

/Users/saiuppalapati/miniforge3/envs/tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: dlopen(/Users/saiuppalapati/miniforge3/envs/tf/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so, 0x0002): tried: '/Users/saiuppalapati/miniforge3/envs/tf/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (section '__DATA/__thread_bss' has a zero-fill section type, but offset field is not zero), '/System/Volumes/Preboot/Cryptexes/OS/Users/saiuppalapati/miniforge3/envs/tf/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (no such file), '/Users/saiuppalapati/miniforge3/envs/tf/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (section '__DATA/__thread_bss' has a zero-fill section type, but offset field is not zero)

## 1.2 - BPE tokenization without a pretrained tokenizer

This simplified implementation learns frequent adjacent-symbol merges from the corpus. It starts with characters plus an end-of-word marker and performs a fixed number of merges.

This is an educational implementation of BPE training; it does not include a complete encoder for arbitrary unseen words.

In [ ]:
import re
from collections import Counter

# Extract lowercase words from the corpus.
words = re.findall(r"\\b[a-z]+\\b", text.lower())

# Represent each word as characters followed by an end-of-word marker.
bpe_vocab = Counter()
for word in words:
    bpe_vocab[" ".join(list(word)) + " </w>"] += 1

num_merges = 20
learned_merges = []

for step in range(num_merges):
    pair_counts = Counter()

    # Count adjacent pairs, weighted by each word's frequency.
    for word_representation, frequency in bpe_vocab.items():
        symbols = word_representation.split()
        for i in range(len(symbols) - 1):
            pair_counts[(symbols[i], symbols[i + 1])] += frequency

    if not pair_counts:
        break

    best_pair = pair_counts.most_common(1)[0][0]
    learned_merges.append(best_pair)

    # Apply the selected merge throughout the vocabulary.
    updated_vocab = Counter()
    for word_representation, frequency in bpe_vocab.items():
        symbols = word_representation.split()
        merged_symbols = []
        i = 0

        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and (symbols[i], symbols[i + 1]) == best_pair
            ):
                merged_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                merged_symbols.append(symbols[i])
                i += 1

        updated_vocab[" ".join(merged_symbols)] += frequency

    bpe_vocab = updated_vocab
    print(f"Merge {step + 1:02d}: {best_pair[0]} + {best_pair[1]}")

# Create a set of the symbols remaining after training.
learned_symbols = set()
for word_representation in bpe_vocab:
    learned_symbols.update(word_representation.split())

# Assign deterministic IDs to the learned symbols.
bpe_token_to_id = {
    symbol: idx for idx, symbol in enumerate(sorted(learned_symbols))
}

print("\nLearned vocabulary entries (first 30):")
for item, frequency in list(bpe_vocab.items())[:30]:
    print(f"{item}: {frequency}")

print("\nLearned symbol vocabulary size:", len(bpe_token_to_id))
print("\nExample token IDs (first 30 symbols):")
print(dict(list(bpe_token_to_id.items())[:30]))

## 2.1 - SentencePiece-based tokenization using a pretrained model

T5 uses a SentencePiece-based tokenizer. This section loads the pretrained `t5-small` tokenizer and displays subword tokens and IDs.

In [ ]:
from transformers import AutoTokenizer

t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")

sentencepiece_tokens = t5_tokenizer.tokenize(text)
sentencepiece_token_ids = t5_tokenizer.convert_tokens_to_ids(sentencepiece_tokens)

print("First 100 SentencePiece-based tokens:")
print(sentencepiece_tokens[:100])

print("\nFirst 100 token IDs:")
print(sentencepiece_token_ids[:100])

print("\nNumber of tokens:", len(sentencepiece_tokens))

## 2.2 - Train a SentencePiece BPE tokenizer

This section trains a new SentencePiece tokenizer on the input corpus. The model and vocabulary files are saved with the prefix `my_bpe`.

In [ ]:
import sentencepiece as spm

# SentencePiece expects a text file as training input.
# If the original file was missing, create one from the fallback corpus.
training_file = Path("input_sub_word_data.txt")
if not training_file.exists():
    training_file.write_text(text, encoding="utf-8")

# SentencePiece requires a vocabulary size large enough for its special tokens
# and the characters present in the corpus. If 200 is too small for a corpus,
# reduce it; for a tiny corpus, a smaller value may be appropriate.
vocab_size = 200

spm.SentencePieceTrainer.train(
    input=str(training_file),
    model_prefix="my_bpe",
    vocab_size=vocab_size,
    model_type="bpe",
    character_coverage=1.0,
    hard_vocab_limit=False
)

sp = spm.SentencePieceProcessor(model_file="my_bpe.model")

sp_tokens = sp.encode(text, out_type=str)
sp_token_ids = sp.encode(text, out_type=int)

print("First 100 SentencePiece tokens:")
print(sp_tokens[:100])

print("\nFirst 100 token IDs:")
print(sp_token_ids[:100])

print("\nNumber of tokens:", len(sp_tokens))
print("Actual vocabulary size:", sp.get_piece_size())
print("\nSaved model: my_bpe.model")
print("Saved vocabulary: my_bpe.vocab")

## 3.1 - POS tagging using spaCy

POS tagging assigns grammatical categories such as noun, verb, adjective, and adverb to tokens. If the English model is not installed, the following cell downloads it.

In [ ]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "spacy", "download", "en_core_web_sm"
    ])
    nlp = spacy.load("en_core_web_sm")

sample_text = (
    "Natural language processing enables computers to understand human language. "
    "Machine learning improves language models."
)

doc = nlp(sample_text)

print(f"{'Token':<18}{'POS Tag':<12}Description")
print("-" * 55)

for token in doc:
    print(f"{token.text:<18}{token.pos_:<12}{spacy.explain(token.pos_) or 'Unknown'}")

## 3.2 - POS tagging using NLTK

NLTK's `pos_tag` generally uses Penn Treebank tags, such as `NN` for a singular noun and `VBG` for a gerund or present participle.

In [ ]:
import nltk

# Download the resources commonly needed by current NLTK versions.
for resource in [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
]:
    nltk.download(resource, quiet=True)

sample_text = (
    "Natural language processing enables computers to understand human language. "
    "Machine learning improves language models."
)

tokens = nltk.word_tokenize(sample_text)
tagged_tokens = nltk.pos_tag(tokens)

print(f"{'Token':<20}{'POS Tag'}")
print("-" * 32)

for token, tag in tagged_tokens:
    print(f"{token:<20}{tag}")

## 4 - Unique tokens, POS descriptions, and frequencies using spaCy

This section ignores punctuation and whitespace, groups tokens without regard to letter case, and reports each unique token's POS tag, description, and frequency.

In [ ]:
from collections import Counter

doc = nlp(text)

# Count each token, ignoring punctuation and whitespace.
token_frequency = Counter(
    token.text.lower()
    for token in doc
    if not token.is_space and not token.is_punct
)

# Keep the POS tag and description for each unique lowercase token.
token_information = {}

for token in doc:
    if token.is_space or token.is_punct:
        continue

    word = token.text.lower()
    if word not in token_information:
        token_information[word] = (
            token.pos_,
            spacy.explain(token.pos_) or "Unknown"
        )

print(f"{'Token':<24}{'POS Tag':<12}{'Description':<28}{'Frequency'}")
print("-" * 78)

for word in sorted(token_information):
    pos_tag, description = token_information[word]
    print(f"{word:<24}{pos_tag:<12}{description:<28}{token_frequency[word]}")

print("\nNumber of unique tokens:", len(token_information))

## Conclusion

- **BPE** learns merges of frequent adjacent symbols to build subword units.
- **SentencePiece** can train a subword tokenizer directly from a text corpus.
- **Token IDs** represent tokens numerically for model input.
- **POS tagging** assigns grammatical labels to tokens.
- **Frequency analysis** counts how often each token occurs.

### Notes
- The pretrained-tokenizer sections require an internet connection the first time they download model files.
- Keep `input_sub_word_data.txt` beside this notebook for the experiment to use your uploaded corpus.
- If the SentencePiece trainer reports that the vocabulary size is too high for a small corpus, the notebook uses `hard_vocab_limit=False` so the final vocabulary may be smaller than the requested size.